# Browse Datasets

Explore the four support/incident datasets for the Review & Judge lab.
Each dataset has a `sample.json` (500 rows) in `datasets/`.

In [ ]:
import json
from pathlib import Path
from collections import Counter

def load_sample(name):
    path = Path(f"datasets/{name}/sample.json")
    return json.loads(path.read_text())

# Load all datasets
bitext = load_sample("bitext_customer_support")
tbueck = load_sample("tobi_bueck_tickets")
cfpb = load_sample("cfpb_complaints")

# Twitter may not be downloaded yet
twitter_path = Path("datasets/twitter_customer_support/sample.json")
twitter = json.loads(twitter_path.read_text()) if twitter_path.exists() else []

print(f"Bitext:       {len(bitext)} items")
print(f"Tobi-Bueck:   {len(tbueck)} items")
print(f"CFPB:         {len(cfpb)} items")
print(f"Twitter:      {len(twitter)} items")

## 1. Bitext Customer Support
Short requests, 27 intents, 11 categories. The easiest dataset.

In [ ]:
# Schema
print("Keys:", list(bitext[0].keys()))
print("\nFirst 5 items:")
for item in bitext[:5]:
    print(f"  [{item['ground_truth_intent']:30s} | {item['category']:12s}] {item['text']}")

In [ ]:
# Intent distribution
intent_counts = Counter(d["ground_truth_intent"] for d in bitext)
print(f"{len(intent_counts)} unique intents\n")
for intent, count in intent_counts.most_common():
    bar = "█" * count
    print(f"  {intent:30s} {count:3d} {bar}")

In [ ]:
# Category distribution
cat_counts = Counter(d["category"] for d in bitext)
print(f"{len(cat_counts)} categories\n")
for cat, count in cat_counts.most_common():
    bar = "█" * (count // 2)
    print(f"  {cat:15s} {count:3d} {bar}")

In [ ]:
# Text length stats
lengths = [len(d["text"]) for d in bitext]
print(f"Text length: min={min(lengths)}, avg={sum(lengths)//len(lengths)}, max={max(lengths)}")

## 2. Tobi-Bueck Support Tickets
Email-style tickets with subject + body, priority, and ticket type.

In [ ]:
# Schema and first examples
print("Keys:", list(tbueck[0].keys()))
print("\nFirst 3 items:\n")
for item in tbueck[:3]:
    print(f"  [{item['ticket_type']} | {item['priority']}]")
    print(f"  {item['text'][:200]}...")
    print()

In [ ]:
# Ticket type × priority distribution
print("By ticket type:")
type_counts = Counter(str(d["ticket_type"]) for d in tbueck)
for t, c in type_counts.most_common():
    print(f"  {t:12s} {c:3d} {'█' * (c // 5)}")

print("\nBy priority:")
pri_counts = Counter(str(d["priority"]) for d in tbueck)
for p, c in pri_counts.most_common():
    print(f"  {p:12s} {c:3d} {'█' * (c // 5)}")

# Text length
lengths = [len(d["text"]) for d in tbueck]
print(f"\nText length: min={min(lengths)}, avg={sum(lengths)//len(lengths)}, max={max(lengths)}")

## 3. CFPB Consumer Complaints
Real consumer complaints, PII-redacted. The hardest dataset with labels.

In [ ]:
# Schema and first examples
print("Keys:", list(cfpb[0].keys()))
print("\nFirst 3 items:\n")
for item in cfpb[:3]:
    print(f"  [{item['product']} | {item['issue']}]")
    print(f"  {item['text'][:250]}...")
    print()

In [ ]:
# Product distribution
prod_counts = Counter(d["product"] for d in cfpb)
print(f"{len(prod_counts)} product categories\n")
for prod, count in prod_counts.most_common():
    bar = "█" * (count // 3)
    print(f"  {prod:55s} {count:3d} {bar}")

# Text length
lengths = [len(d["text"]) for d in cfpb]
print(f"\nText length: min={min(lengths)}, avg={sum(lengths)//len(lengths)}, max={max(lengths)}")

## 4. Twitter Customer Support
Real tweets — informal, short, noisy. No labels.

In [ ]:
if not twitter:
    print("Twitter dataset not downloaded. Run: uv run python download_datasets.py")
else:
    print("Keys:", list(twitter[0].keys()))
    print(f"\n10 random tweets:\n")
    import random
    random.seed(42)
    for item in random.sample(twitter, 10):
        print(f"  {item['text'][:120]}")
        print()
    
    lengths = [len(d["text"]) for d in twitter]
    print(f"Text length: min={min(lengths)}, avg={sum(lengths)//len(lengths)}, max={max(lengths)}")

## Cross-dataset comparison
Same metrics across all four datasets.

In [ ]:
datasets = {
    "Bitext": bitext,
    "Tobi-Bueck": tbueck,
    "CFPB": cfpb,
    "Twitter": twitter,
}

print(f"{'Dataset':15s} {'Count':>6s} {'Min len':>8s} {'Avg len':>8s} {'Max len':>8s} {'Labels':>10s}")
print("-" * 65)
for name, data in datasets.items():
    if not data:
        print(f"{name:15s} {'N/A':>6s}")
        continue
    lengths = [len(d["text"]) for d in data]
    # Count label fields (keys beyond id, source, text)
    label_keys = [k for k in data[0] if k not in ("id", "source", "text")]
    print(f"{name:15s} {len(data):6d} {min(lengths):8d} {sum(lengths)//len(lengths):8d} {max(lengths):8d} {', '.join(label_keys) or 'none':>10s}")

## Browse any item
Change the dataset and index to explore.

In [ ]:
# Change these to browse
DATASET = "cfpb"        # bitext, tbueck, cfpb, twitter
INDEX = 42              # 0-499

lookup = {"bitext": bitext, "tbueck": tbueck, "cfpb": cfpb, "twitter": twitter}
item = lookup[DATASET][INDEX]

print(f"--- {DATASET} item #{INDEX} ---\n")
for key, val in item.items():
    if key == "text":
        print(f"\ntext:\n{val}")
    else:
        print(f"{key}: {val}")